In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Cell 1 — Install dependencies
!pip install -q chromadb rank-bm25 sentence-transformers tiktoken \
             beautifulsoup4 requests bert-score accelerate bitsandbytes

In [ ]:
pip install -q bitsandbytes

In [ ]:
# Cell 2 — Check GPU
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("CPU only — enable GPU in Runtime settings for faster inference")

In [ ]:
# Cell 3 — Load Mistral 7B
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import torch

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
) if torch.cuda.is_available() else None

print(f"Loading {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.1,
)
print("Mistral loaded!")

In [ ]:
# Cell 4 — LLM helper
def llm_chat(prompt: str, max_tokens: int = 512) -> str:
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    out = llm_pipeline(formatted, max_new_tokens=max_tokens,
                       pad_token_id=tokenizer.eos_token_id)
    return out[0]["generated_text"][len(formatted):].strip()

# Quick test
print(llm_chat("Say hello in one sentence.", max_tokens=40))

In [ ]:
# Cell 5 — Chunker
import hashlib, re, json
from dataclasses import dataclass, field

@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    text: str
    source_url: str = ""
    questions: list = field(default_factory=list)

def chunk_document(doc_id, text, source_url="", chunk_size=800, overlap=100):
    words = text.split()
    chunks, step = [], chunk_size - overlap
    for i, start in enumerate(range(0, len(words), step)):
        chunk_text = " ".join(words[start:start + chunk_size])
        if len(chunk_text.strip()) < 200:
            continue
        cid = hashlib.md5(f"{doc_id}_{i}".encode()).hexdigest()[:12]
        chunks.append(Chunk(chunk_id=cid, doc_id=doc_id,
                            text=chunk_text, source_url=source_url))
        if start + chunk_size >= len(words):
            break
    print(f"[Chunker] {doc_id}: {len(chunks)} chunks")
    return chunks

In [ ]:
# Cell 6 — Question generator
QUESTION_PROMPT = '''You are generating questions strictly from the given text chunk.

Task:
Create a set of high-quality questions based ONLY on the provided text.

STRICT RULES:
- Every question MUST be directly answerable from the text.
- Do NOT use outside knowledge.
- Do NOT infer or assume anything not explicitly stated.
- Do NOT generate generic questions.
- Focus only on important and meaningful information present in the text.
- Avoid overemphasizing any specific type of detail (e.g., numbers, names, etc.).
- Avoid duplicates.

Output Rules:
- Return ONLY a valid JSON array of strings.
- No explanation or extra text.
- No trailing commas.

If the text contains limited information, generate fewer questions.
If no valid questions can be formed, return [].

Text:
"""{text}"""'''

def generate_questions(chunk, max_q=6):
    prompt = QUESTION_PROMPT.format(text=chunk.text[:1500])
    try:
        raw = llm_chat(prompt, max_tokens=400)
        match = re.search(r"\[.*?\]", raw, re.DOTALL)
        if not match:
            print(f"  [QGen] No JSON found for {chunk.chunk_id}")
            return []
        questions = json.loads(match.group())
        return [q.strip() for q in questions if isinstance(q, str)][:max_q]
    except Exception as e:
        print(f"  [QGen] Failed: {e}")
        return []

In [ ]:
# Cell 7 — Embedding model + ChromaDB (in-memory)
from sentence_transformers import SentenceTransformer
import chromadb

print("Loading embedding model BAAI/bge-large-en-v1.5 ...")
embed_model = SentenceTransformer("BAAI/bge-large-en-v1.5")
print("Embedding model ready")

chroma_client = chromadb.Client()  # in-memory, no disk needed in Colab/Kaggle
collection = chroma_client.get_or_create_collection(
    name="quim_rag",
    metadata={"hnsw:space": "cosine"},
)
print(f"ChromaDB ready")

In [ ]:
# Cell 8 — Index helpers
def add_chunk_to_index(chunk):
    if not chunk.questions:
        return
    embeddings = embed_model.encode(chunk.questions, normalize_embeddings=True).tolist()
    ids, docs, metas, embs = [], [], [], []
    for q_idx, (q, emb) in enumerate(zip(chunk.questions, embeddings)):
        ids.append(f"{chunk.chunk_id}_q{q_idx}")
        docs.append(q)
        metas.append({"chunk_id": chunk.chunk_id, "doc_id": chunk.doc_id,
                      "source_url": chunk.source_url})
        embs.append(emb)
    collection.upsert(ids=ids, documents=docs, metadatas=metas, embeddings=embs)

def query_index(query_text, top_k=9):
    count = collection.count()
    if count == 0:
        return []
    q_emb = embed_model.encode([query_text], normalize_embeddings=True).tolist()
    results = collection.query(
        query_embeddings=q_emb,
        n_results=min(top_k, count),
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    if results["ids"] and results["ids"][0]:
        for doc, meta, dist in zip(results["documents"][0],
                                   results["metadatas"][0],
                                   results["distances"][0]):
            hits.append({"question": doc, "chunk_id": meta["chunk_id"],
                         "source_url": meta.get("source_url", ""),
                         "score": 1.0 - dist})
    return hits

In [ ]:
# Cell 9 — BM25 + Hybrid RRF
from rank_bm25 import BM25Okapi

all_chunks = {}   # chunk_id -> Chunk
bm25_index = None
bm25_order = []

def build_bm25():
    global bm25_index, bm25_order
    chunks = list(all_chunks.values())
    if not chunks:
        bm25_index = None
        return
    bm25_order = [c.chunk_id for c in chunks]
    bm25_index = BM25Okapi([c.text.lower().split() for c in chunks])
    print(f"[BM25] Built over {len(chunks)} chunks")

def hybrid_retrieve(query, top_k=3, rrf_k=60, alpha=0.5):
    dense_hits = query_index(query, top_k=top_k * 3)
    dense_scores = {}
    for rank, hit in enumerate(dense_hits):
        cid = hit["chunk_id"]
        if cid not in dense_scores:
            dense_scores[cid] = 1.0 / (rrf_k + rank + 1)

    sparse_scores = {}
    if bm25_index and bm25_order:
        raw = bm25_index.get_scores(query.lower().split())
        ranked = sorted(enumerate(raw), key=lambda x: x[1], reverse=True)[:top_k * 3]
        for rank, (idx, _) in enumerate(ranked):
            sparse_scores[bm25_order[idx]] = 1.0 / (rrf_k + rank + 1)

    all_ids = set(dense_scores) | set(sparse_scores)
    fused = sorted(
        [(cid, alpha * dense_scores.get(cid, 0) + (1 - alpha) * sparse_scores.get(cid, 0))
         for cid in all_ids],
        key=lambda x: x[1], reverse=True
    )
    results = []
    for cid, score in fused[:top_k]:
        chunk = all_chunks.get(cid)
        if not chunk:
            continue
        matched_q = next((h["question"] for h in dense_hits if h["chunk_id"] == cid), "")
        results.append({"chunk_id": cid, "question": matched_q,
                        "score": score, "source_url": chunk.source_url})
    return results

In [ ]:
def bm25_retrieve(query, top_k=5):
    if bm25_index is None:
        return []

    scores = bm25_index.get_scores(query.lower().split())
    ranked = sorted(
        enumerate(scores),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    hits = []
    for idx, score in ranked:
        chunk_id = bm25_order[idx]
        chunk = all_chunks.get(chunk_id)
        if chunk:
            hits.append({
                "chunk_id": chunk_id,
                "score": float(score),
                "source_url": chunk.source_url
            })
    return hits


def dense_retrieve(query, top_k=5):
    q_emb = embed_model.encode(
        [query],
        normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings=q_emb,
        n_results=top_k,
        include=["metadatas", "distances"]
    )

    hits = []
    for meta, dist in zip(
        results["metadatas"][0],
        results["distances"][0]
    ):
        hits.append({
            "chunk_id": meta["chunk_id"],
            "score": 1.0 - dist,
            "source_url": meta.get("source_url", "")
        })
    return hits

def hybrid_retrieve(query, top_k=5, alpha=0.5):
    bm25_hits = bm25_retrieve(query, top_k=top_k * 2)
    dense_hits = dense_retrieve(query, top_k=top_k * 2)

    scores = {}

    for rank, hit in enumerate(bm25_hits):
        scores.setdefault(hit["chunk_id"], 0.0)
        scores[hit["chunk_id"]] += alpha / (rank + 1)

    for rank, hit in enumerate(dense_hits):
        scores.setdefault(hit["chunk_id"], 0.0)
        scores[hit["chunk_id"]] += (1 - alpha) / (rank + 1)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    results = []
    for cid, score in ranked:
        chunk = all_chunks.get(cid)
        if chunk:
            results.append({
                "chunk_id": cid,
                "score": score,
                "source_url": chunk.source_url
            })
    return results

def quimrag_retrieve(query, top_k=5):
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    proto_id = int(kmeans.predict(q_emb)[0])

    candidate_qids = inverted_index.get(proto_id, [])
    if not candidate_qids:
        return []

    raw = collection.query(
        query_embeddings=q_emb.tolist(),
        n_results=min(50, collection.count()),
        include=["metadatas", "distances"],
    )

    hits = []
    for qid, meta, dist in zip(
        raw["ids"][0],
        raw["metadatas"][0],
        raw["distances"][0]
    ):
        if qid in candidate_qids:
            hits.append({
                "chunk_id": meta["chunk_id"],
                "score": 1.0 - dist,
                "source_url": meta.get("source_url", "")
            })
        if len(hits) >= top_k:
            break

    return hits

In [ ]:
def ask(query, retriever="quimrag"):
    query = rewrite_query(query)

    if retriever == "bm25":
        hits = bm25_retrieve(query)

    elif retriever == "dense":
        hits = dense_retrieve(query)

    elif retriever == "hybrid":
        hits = hybrid_retrieve(query)

    elif retriever == "quimrag":
        hits = quimrag_retrieve(query)

    else:
        raise ValueError(f"Unknown retriever: {retriever}")

    if not hits:
        return {
            "answer": "No relevant context found.",
            "contexts": []
        }

    answer = generate_answer(query, hits)

    return {
        "answer": answer,
        "contexts": hits,
        "retriever": retriever
    }

In [ ]:

for retriever in ["bm25", "dense", "hybrid", "quimrag"]:
    print("\n=== Evaluating:", retriever)
    result = ask("What does HTML stand for?", retriever=retriever)
    print(result["answer"])


In [ ]:
BASELINE_QA = [
    {
        "question": "What does HTML stand for?",
        "answer": "HTML stands for HyperText Markup Language."
    },
    {
        "question": "What is a web browser?",
        "answer": "A web browser is a software application used to access and display web content."
    },
    {
        "question": "What is a browser engine?",
        "answer": "A browser engine is a software component used to render web pages."
    },
    {
        "question": "Which browser engine is used by Google Chrome?",
        "answer": "Google Chrome uses the Blink browser engine."
    },
    {
        "question": "What is a table in information organization?",
        "answer": "A table is a structured arrangement of data in rows and columns."
    }
]

In [ ]:
def evaluate_retrievers_small_set(qa_set, retrievers):
    results = {}

    for retriever in retrievers:
        print(f"\n=== Evaluating retriever: {retriever}")
        candidates, references = [], []

        for qa in qa_set:
            result = ask(qa["question"], retriever=retriever)
            candidates.append(result["answer"])
            references.append(qa["answer"])

            print(f"Q: {qa['question']}")
            print(f"A: {result['answer'][:100]}...\n")

        # BERTScore (fast & suitable for small sets)
        P, R, F1 = bert_score_fn(
            candidates,
            references,
            lang="en",
            device="cuda"  # if GPU is available
        )

        results[retriever] = {
            "bert_precision": P.mean().item(),
            "bert_recall": R.mean().item(),
            "bert_f1": F1.mean().item(),
        }

    return results


In [ ]:
baseline_results = evaluate_retrievers_small_set(
    BASELINE_QA,
    retrievers=["bm25", "dense", "hybrid", "quimrag"]
)

baseline_results

In [ ]:
# Cell 10 — Cross-encoder reranker
from sentence_transformers import CrossEncoder

print("Loading cross-encoder ...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder ready")

def rerank(query, hits, top_k=3):
    if not hits:
        return hits
    pairs = [(query, all_chunks[h["chunk_id"]].text[:512]
              if h["chunk_id"] in all_chunks else h["question"])
             for h in hits]
    scores = reranker.predict(pairs).tolist()
    for hit, score in zip(hits, scores):
        hit["rerank_score"] = score
    return sorted(hits, key=lambda h: h["rerank_score"], reverse=True)[:top_k]

In [ ]:
# Cell 11 — Query rewriter
REWRITE_PROMPT = '''Rewrite this question to be more specific and self-contained
for searching a university knowledge base.
Return ONLY the rewritten question, nothing else.

Question: {query}
Rewritten:'''

def rewrite_query(query):
    if len(query.split()) > 10:
        return query
    try:
        result = llm_chat(REWRITE_PROMPT.format(query=query), max_tokens=60)
        rewritten = result.strip().split("\n")[0]
        print(f"[Rewriter] '{query}'  ->  '{rewritten}'")
        return rewritten
    except Exception:
        return query

In [ ]:
from sklearn.cluster import MiniBatchKMeans
import numpy as np

NUM_PROTOTYPES = 256

def build_prototypes():
    all_q_embeddings = []
    q_ids = []

    for cid, chunk in all_chunks.items():
        if not chunk.questions:
            continue
        emb = embed_model.encode(chunk.questions, normalize_embeddings=True)
        for i, e in enumerate(emb):
            all_q_embeddings.append(e)
            q_ids.append(f"{cid}_q{i}")

    X = np.vstack(all_q_embeddings)

    kmeans = MiniBatchKMeans(
        n_clusters=NUM_PROTOTYPES,
        batch_size=1024,
        random_state=42
    )
    labels = kmeans.fit_predict(X)

    print(f"[Prototypes] Built {NUM_PROTOTYPES} prototypes")

    return kmeans, q_ids, labels

In [ ]:
from collections import defaultdict

def build_inverted_index(q_ids, labels):
    inverted_index = defaultdict(list)

    for qid, proto_id in zip(q_ids, labels):
        inverted_index[int(proto_id)].append(qid)

    print("[Inverted Index] Ready")
    return dict(inverted_index)


In [ ]:
def prototype_retrieve(query, kmeans, inverted_index, top_k=9):
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    proto_id = int(kmeans.predict(q_emb)[0])

    candidate_qids = inverted_index.get(proto_id, [])
    if not candidate_qids:
        return []

    results = collection.query(
        query_embeddings=q_emb.tolist(),
        n_results=min(top_k, len(candidate_qids)),
        where={"id": {"$in": candidate_qids}},
        include=["documents", "metadatas", "distances"],
    )

    hits = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        hits.append({
            "question": doc,
            "chunk_id": meta["chunk_id"],
            "score": 1.0 - dist,
            "source_url": meta.get("source_url", "")
        })

    return hits

In [ ]:
# Cell 12 — Ingest document
def ingest_document(doc_id, text, source_url=""):
    chunks = chunk_document(doc_id, text, source_url)
    indexed = 0
    for chunk in chunks:
        if chunk.chunk_id in all_chunks:
            print(f"  Chunk {chunk.chunk_id} already indexed, skipping")
            continue
        print(f"  Generating questions for chunk {chunk.chunk_id} ...")
        chunk.questions = generate_questions(chunk)
        if not chunk.questions:
            print(f"  No questions generated, skipping")
            continue
        print(f"  Questions: {chunk.questions}")
        add_chunk_to_index(chunk)
        all_chunks[chunk.chunk_id] = chunk
        indexed += 1
    build_bm25()
    print(f"[Ingest] {indexed} new chunks indexed for '{doc_id}'")
    return indexed

In [ ]:
# Cell 13 — Answer generator
ANSWER_PROMPT = '''You are a helpful university assistant.
Answer the question using ONLY the context below.
Include the source URL if available.
If the context does not contain the answer, say: "I cannot find this information."

Context:
{context}

Question: {question}

Answer:'''

def generate_answer(query, hits):
    blocks = []
    for i, hit in enumerate(hits, 1):
        chunk = all_chunks.get(hit["chunk_id"])
        if not chunk:
            continue
        block = f"[{i}] {chunk.text[:600]}"
        if chunk.source_url:
            block += f"\nSource: {chunk.source_url}"
        blocks.append(block)
    if not blocks:
        return "No relevant context found."
    return llm_chat(ANSWER_PROMPT.format(context="\n\n".join(blocks), question=query),
                    max_tokens=400)

In [ ]:
# Cell 14 — ask() full pipeline
def ask(query, use_rewriter=True, use_reranker=True):
    rewritten = rewrite_query(query) if use_rewriter else query
    hits = prototype_retrieve(
        rewritten,
        kmeans,   
        inverted_index,
        top_k=9
    )

    if not hits or max(h.get("rerank_score", h["score"]) for h in hits) < 0.3:
        return {
        "answer": "I don't have enough information to answer this question.",
        "query": query,
        "rewritten_query": rewritten,
        "contexts": []
    }
    hits = rerank(rewritten, hits, top_k=3) if use_reranker else hits[:3]
    answer = generate_answer(query, hits)
    return {
        "answer": answer,
        "query": query,
        "rewritten_query": rewritten,
        "contexts": [
            {"chunk_id": h["chunk_id"],
             "score": round(h.get("rerank_score", h["score"]), 4),
             "source_url": h["source_url"],
             "snippet": all_chunks[h["chunk_id"]].text[:150] + "..."
             if h["chunk_id"] in all_chunks else ""}
            for h in hits
        ],
    }

In [ ]:
# # Cell 15 — Sample NDSU documents
# SAMPLE_DOCS = [
#     {
#         "doc_id": "cs_program",
#         "url": "https://catalog.ndsu.edu/undergraduate/computer-science/",
#         "text": (
#             "Computer Science B.S. at NDSU. Core Requirements 54 credits: "
#             "CSCI 160 Computer Science I 3 credits, CSCI 161 Computer Science II 3 credits, "
#             "CSCI 336 Computer Organization 3 credits, CSCI 366 Algorithms and Data Structures 3 credits, "
#             "CSCI 374 Operating Systems 3 credits, CSCI 474 Software Engineering 3 credits, "
#             "CSCI 489 Senior Capstone 3 credits, MATH 165 Calculus I 4 credits, "
#             "MATH 166 Calculus II 4 credits, STAT 330 Statistics 3 credits. "
#             "Graduation requires 122 total credits, minimum GPA 2.0 in all CSCI courses. "
#             "Transfer students may transfer up to 64 credits and must complete 30 at NDSU."
#         ),
#     },
#     {
#         "doc_id": "career_advising",
#         "url": "https://career-advising.ndsu.edu/contact/",
#         "text": (
#             "NDSU Career and Advising Center. Location 306 Ceres Hall Fargo ND 58108. "
#             "Phone 701-231-7111. Email ndsu.cac@ndsu.edu. "
#             "Hours Monday to Friday 9 AM to 4 PM. "
#             "Services include resume review, mock interviews, job search via Handshake, "
#             "career fairs every fall and spring semester, graduate school advising. "
#             "Drop-in hours Monday to Thursday 10 AM to 2 PM no appointment needed. "
#             "Full 30-minute appointments available via Handshake or by calling 701-231-7111."
#         ),
#     },
# ]

# for doc in SAMPLE_DOCS:
#     ingest_document(doc["doc_id"], doc["text"], doc["url"])

# print(f"\nTotal chunks indexed: {len(all_chunks)}")

In [ ]:
# # Cell 16 — Test questions
# test_questions = [
#     "contact career center",
#     "CS graduation requirements",
#     "how many credits to graduate",
# ]

# for q in test_questions:
#     print("=" * 60)
#     print(f"Q: {q}")
#     result = ask(q)
#     print(f"Rewritten : {result['rewritten_query']}")
#     print(f"\nAnswer:\n{result['answer']}")
#     print("\nSources:")
#     for ctx in result["contexts"]:
#         print(f"  [{ctx['score']}] {ctx['source_url']}")
#     print()

In [ ]:
import urllib.robotparser as urobot
from urllib.parse import urljoin

def is_allowed(url, user_agent="QuIMRAGBot"):
    robots_url = urljoin(url, "/robots.txt")
    rp = urobot.RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
        return rp.can_fetch(user_agent, url)
    except Exception:
        # If robots.txt cannot be read, fail open (common academic practice)
        return True

In [ ]:
def extract_tables(soup):
    tables = []
    for table in soup.find_all("table"):
        rows = []
        headers = [th.get_text(strip=True) for th in table.find_all("th")]
        
        for tr in table.find_all("tr"):
            cells = [td.get_text(strip=True) for td in tr.find_all("td")]
            if cells:
                row_text = ", ".join(
                    f"{headers[i]}: {cells[i]}" 
                    for i in range(min(len(headers), len(cells)))
                )
                rows.append(row_text)
        
        if rows:
            tables.append("Table Info:\n" + "\n".join(rows))
    return tables

In [ ]:
!pip install pytesseract pillow

In [ ]:
import cv2
import numpy as np

def extract_image_text(img_url):
    try:
        resp = requests.get(img_url, timeout=10)
        img = Image.open(BytesIO(resp.content)).convert("RGB")

        # Convert to OpenCV format
        img_np = np.array(img)
        gray = cv2.cvtColor(img_np, cv2.COLOR_BGR2GRAY)

        # --- Preprocessing ---
        gray = cv2.medianBlur(gray, 3)          # reduce noise
        _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

        # --- OCR config ---
        config = "--oem 3 --psm 6"

        text = pytesseract.image_to_string(thresh, config=config)

        # --- Clean text ---
        text = re.sub(r'[^a-zA-Z0-9.,:% ]+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()

        return text if len(text) > 20 else None

    except Exception:
        return None


In [ ]:
!pip install PyPDF2

In [ ]:
import PyPDF2

def extract_pdf_text(pdf_url):
    try:
        resp = requests.get(pdf_url, timeout=10)
        reader = PyPDF2.PdfReader(BytesIO(resp.content))
        text = []
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text.append(page_text)
        return "\n".join(text) if text else None
    except Exception:
        return None

In [ ]:
import requests, time, re, hashlib
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
def scrape_site(seed_url, max_pages=10):
    domain = urlparse(seed_url).netloc
    visited, queue, pages = set(), [seed_url], []
    session = requests.Session()
    session.headers["User-Agent"] = "QuIMRAGBot"

    while queue and len(pages) < max_pages:
        url = queue.pop(0)
        norm = url.rstrip("/").lower().split("?")[0]

        if norm in visited:
            continue
        visited.add(norm)

        try:
            resp = session.get(url, timeout=10)
            if resp.status_code != 200:
                continue

            soup = BeautifulSoup(resp.text, "html.parser")

            # Remove noise
            for tag in soup(["header", "footer", "nav", "script", "style"]):
                tag.decompose()

            main = soup.find("main") or soup.find("body")
            if not main:
                continue

            # --- TEXT ---
            text = re.sub(r"\s+", " ", main.get_text()).strip()

            # --- TABLES ---
            table_texts = extract_tables(soup)

            # --- IMAGES ---
            image_texts = []
            for img in soup.find_all("img", src=True):
                img_url = urljoin(url, img["src"])
                ocr_text = extract_image_text(img_url)
                if ocr_text:
                    image_texts.append(f"Image Text: {ocr_text}")

            # --- PDFs ---
            pdf_texts = []
            for a in soup.find_all("a", href=True):
                if a["href"].lower().endswith(".pdf"):
                    pdf_url = urljoin(url, a["href"])
                    pdf_text = extract_pdf_text(pdf_url)
                    if pdf_text:
                        pdf_texts.append(f"PDF Content: {pdf_text}")

            # --- COMBINE ALL ---
            final_text = "\n\n".join(
                [text] + table_texts + image_texts + pdf_texts
            )

            if len(final_text) < 300:
                continue

            doc_id = hashlib.md5(url.encode()).hexdigest()[:10]
            pages.append({
                "doc_id": doc_id,
                "url": url,
                "text": final_text,
                "timestamp": time.time()
            })

            print(f"✅ Scraped enriched content: {url}")

            for link in soup.find_all("a", href=True):
                full = urljoin(url, link["href"]).split("#")[0]
                if urlparse(full).netloc == domain:
                    queue.append(full)

            time.sleep(0.5)

        except Exception as e:
            print(f"Error: {e}")

    return pages

In [ ]:
pages = scrape_site("https://en.wikipedia.org/wiki/Table_(information)", max_pages=5)
for page in pages:
    ingest_document(page["doc_id"], page["text"], page["url"])
print("Scraper ready — uncomment above lines to use")

In [ ]:

print("Chunks indexed:", len(all_chunks))
print("Questions indexed:", collection.count())


In [ ]:
from sklearn.cluster import MiniBatchKMeans
import numpy as np

def build_prototypes(max_prototypes=256, min_prototypes=2):
    all_q_embeddings = []
    q_ids = []

    for cid, chunk in all_chunks.items():
        if not chunk.questions:
            continue
        emb = embed_model.encode(chunk.questions, normalize_embeddings=True)
        for i, e in enumerate(emb):
            all_q_embeddings.append(e)
            q_ids.append(f"{cid}_q{i}")

    if not all_q_embeddings:
        raise RuntimeError("No question embeddings found. Ingest data first.")

    X = np.vstack(all_q_embeddings)
    n_samples = X.shape[0]

    # ✅ Adaptive prototype count
    n_clusters = min(max_prototypes, max(min_prototypes, n_samples // 2))

    print(f"[Prototypes] Using {n_clusters} prototypes for {n_samples} questions")

    kmeans = MiniBatchKMeans(
        n_clusters=n_clusters,
        batch_size=min(1024, n_samples),
        random_state=42
    )

    labels = kmeans.fit_predict(X)

    return kmeans, q_ids, labels


In [ ]:
# ✅ Build prototypes and inverted index (MANDATORY)
kmeans, q_ids, labels = build_prototypes()
inverted_index = build_inverted_index(q_ids, labels)

print("✅ Prototypes and inverted index initialized")

In [ ]:
print("Prototype buckets:", len(inverted_index))
print("Example bucket size:", len(next(iter(inverted_index.values()))))

# Try raw Chroma query WITHOUT prototype filter
print("Raw Chroma count:", collection.count())
test = collection.query(
    query_embeddings=embed_model.encode(["html"], normalize_embeddings=True),
    n_results=5
)
print("Raw Chroma results:", test["documents"][0])


In [ ]:
def prototype_retrieve(query, kmeans, inverted_index, top_k=9):
    # 1️⃣ Embed query
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    
    # 2️⃣ Predict prototype
    proto_id = int(kmeans.predict(q_emb)[0])

    candidate_qids = inverted_index.get(proto_id, [])
    if not candidate_qids:
        return []

    # 3️⃣ Get MORE results from Chroma (no ID filter!)
    raw = collection.query(
        query_embeddings=q_emb.tolist(),
        n_results=min(50, collection.count()),  # pull larger pool
        include=["documents", "metadatas", "distances"],
    )

    hits = []
    for doc, meta, dist, qid in zip(
        raw["documents"][0],
        raw["metadatas"][0],
        raw["distances"][0],
        raw["ids"][0],
    ):
        # ✅ FILTER MANUALLY by prototype bucket
        if qid in candidate_qids:
            hits.append({
                "question": doc,
                "chunk_id": meta["chunk_id"],
                "source_url": meta.get("source_url", ""),
                "score": 1.0 - dist,
            })
        if len(hits) >= top_k:
            break

    return hits

In [ ]:
def interactive_qa():
    print("=" * 60)
    print("QuIM‑RAG Question Answering System")
    print("Type 'exit' to quit.")
    print("=" * 60)

    while True:
        query = input("\nEnter your question: ").strip()
        if query.lower() in ["exit", "quit", "q"]:
            print("Exiting QuIM‑RAG. Goodbye!")
            break

        print("\nThinking... ✅\n")
        result = ask(query)

        print("Answer:")
        print("-" * 40)
        print(result["answer"])
        print("-" * 40)

        if result["contexts"]:
            print("\nSource Contexts:")
            for i, ctx in enumerate(result["contexts"], 1):
                print(f"{i}. Chunk ID: {ctx['chunk_id']}")
                print(f"   Score: {ctx['score']}")
                if ctx.get("source_url"):
                    print(f"   URL: {ctx['source_url']}")
        else:
            print("\n(No supporting context found)")

In [ ]:
interactive_qa()

In [ ]:
# import requests, time, re, hashlib
# from bs4 import BeautifulSoup
# from urllib.parse import urljoin, urlparse

# def scrape_site(seed_url, max_pages=10):
#     domain = urlparse(seed_url).netloc
#     visited, queue, pages = set(), [seed_url], []
#     session = requests.Session()
#     session.headers["User-Agent"] = "QuIMRAGBot"

#     while queue and len(pages) < max_pages:
#         url = queue.pop(0)
#         norm = url.rstrip("/").lower().split("?")[0]

#         if norm in visited:
#             continue
#         visited.add(norm)

#         # ✅ robots.txt check — THIS IS WHERE IT GOES
#         if not is_allowed(url, user_agent="QuIMRAGBot"):
#             print(f"  Skipped (robots.txt): {url}")
#             continue

#         try:
#             resp = session.get(url, timeout=10)
#             if resp.status_code != 200:
#                 continue

#             soup = BeautifulSoup(resp.text, "html.parser")

#             title = soup.find("title")
#             if title and "404" in title.get_text().lower():
#                 continue

#             for tag in soup(["header", "footer", "nav", "script", "style"]):
#                 tag.decompose()

#             main = soup.find("main") or soup.find("body")
#             if not main:
#                 continue

#             text = re.sub(r"\s+", " ", main.get_text(separator=" ")).strip()
#             if len(text) < 250:
#                 continue

#             doc_id = hashlib.md5(url.encode()).hexdigest()[:10]
#             pages.append({
#                 "doc_id": doc_id,
#                 "url": url,
#                 "text": text,
#                 "timestamp": time.time()
#             })

#             print(f"  Scraped [{len(pages)}]: {url}")

#             for a in soup.find_all("a", href=True):
#                 full = urljoin(url, a["href"]).split("#")[0]
#                 if urlparse(full).netloc == domain:
#                     queue.append(full)

#             time.sleep(0.5)

#         except Exception as e:
#             print(f"  Error {url}: {e}")

#     return pages

In [ ]:
# # Cell 17 — Web scraper (optional)
# import requests, time
# from bs4 import BeautifulSoup
# from urllib.parse import urljoin, urlparse

# def scrape_site(seed_url, max_pages=10):
#     domain = urlparse(seed_url).netloc
#     visited, queue, pages = set(), [seed_url], []
#     session = requests.Session()
#     session.headers["User-Agent"] = "Mozilla/5.0 (QuIMRAGBot)"

#     while queue and len(pages) < max_pages:
#         url = queue.pop(0)
#         norm = url.rstrip("/").lower().split("?")[0]
#         if norm in visited:
#             continue
#         visited.add(norm)
#         try:
#             resp = session.get(url, timeout=10)
#             if resp.status_code != 200:
#                 continue
#             soup = BeautifulSoup(resp.text, "html.parser")
#             title = soup.find("title")
#             if title and "404" in title.get_text():
#                 continue
#             for tag in soup(["header", "footer", "nav", "script", "style"]):
#                 tag.decompose()
#             main = soup.find("main") or soup.find("body")
#             text = re.sub(r"\s+", " ", main.get_text(separator=" ")).strip()
#             if len(text) < 250:
#                 continue
#             doc_id = hashlib.md5(url.encode()).hexdigest()[:10]
#             pages.append({"doc_id": doc_id, "url": url, "text": text})
#             print(f"  Scraped [{len(pages)}]: {url}")
#             for a in soup.find_all("a", href=True):
#                 full = urljoin(url, a["href"]).split("#")[0]
#                 if urlparse(full).netloc == domain and full not in visited:
#                     queue.append(full)
#             time.sleep(0.5)
#         except Exception as e:
#             print(f"  Error {url}: {e}")
#     return pages

# # Uncomment to scrape a real site:
# pages = scrape_site("https://career-advising.ndsu.edu", max_pages=10)
# for page in pages:
#     ingest_document(page["doc_id"], page["text"], page["url"])
# print("Scraper ready — uncomment above lines to use")

In [ ]:
# # Cell 18 — BERTScore evaluation
# from bert_score import score as bert_score_fn

# ground_truth_qa = [
#     {
#         "question": "who are the members of NDSUIntern spotlights",
#         "answer": "Elham Shirvanihosseini,David Owuor,Katelynn Riley,Alex Nelson"
#     },
#     {
#         "question": "Date of first headshot event",
#         "answer": "15/04/2025or april15 ,2026"
#     },
# ]
# #
# candidates, references = [], []
# for gt in ground_truth_qa:
#     result = ask(gt["question"])
#     candidates.append(result["answer"])
#     references.append(gt["answer"])
#     print(f"Q: {gt['question']}")
#     print(f"Generated: {result['answer']}\n")

# P, R, F1 = bert_score_fn(candidates, references, lang="en", verbose=False)
# print("=" * 40)
# print(f"BERTScore Precision : {P.mean().item():.4f}")
# print(f"BERTScore Recall    : {R.mean().item():.4f}")
# print(f"BERTScore F1        : {F1.mean().item():.4f}")

In [ ]:
# Cell 18 — Ground truth QA pairs + generate answers
import re, json as _json
import numpy as np
from dataclasses import dataclass, field as dc_field

@dataclass
class QAPair:
    question: str
    ground_truth: str
    generated_answer: str = ""
    retrieved_contexts: list = dc_field(default_factory=list)

GROUND_TRUTH_QA = [
    {
        "question": "What does HTML stand for?",
        "answer": "HTML stands for HyperText Markup Language."
    },
    {
        "question": "What is HTML primarily used for?",
        "answer": "HTML is used to structure and present content on the World Wide Web."
    },
    {
        "question": "Is HTML a programming language or a markup language?",
        "answer": "HTML is a markup language."
    },
    {
        "question": "What file extension is commonly used for HTML documents?",
        "answer": "HTML documents commonly use the .html or .htm file extension."
    },
    {
        "question": "What is a web browser?",
        "answer": "A web browser is a software application used to access and display web content."
    },
    {
        "question": "Name some commonly used web browsers.",
        "answer": "Commonly used web browsers include Google Chrome, Mozilla Firefox, Safari, and Microsoft Edge."
    },
    {
        "question": "What protocols do web browsers typically support?",
        "answer": "Web browsers typically support HTTP and HTTPS protocols."
    },
    {
        "question": "What is the main function of a browser engine?",
        "answer": "The main function of a browser engine is to render web content such as HTML and CSS into a visual format."
    },
    {
        "question": "What is a browser engine?",
        "answer": "A browser engine is a core software component of a web browser that renders webpages."
    },
    {
        "question": "Name some popular browser engines.",
        "answer": "Popular browser engines include Blink, WebKit, and Gecko."
    },
    {
        "question": "Which browser engine is used by Google Chrome?",
        "answer": "Google Chrome uses the Blink browser engine."
    },
    {
        "question": "How does a web browser display HTML content?",
        "answer": "A web browser uses its browser engine to interpret HTML and render it as a visual webpage."
    },
    {
        "question": "What is the relationship between HTML and a browser engine?",
        "answer": "HTML defines the structure of a webpage, and the browser engine renders it for display."
    },
    {
        "question": "What is a table in information organization?",
        "answer": "A table is a structured arrangement of data organized into rows and columns."
    },
    {
        "question": "What are the basic components of a table?",
        "answer": "The basic components of a table are rows, columns, and cells."
    },
    {
        "question": "Why are tables used to present information?",
        "answer": "Tables are used to organize, compare, and clearly present data."
    },
    {
        "question": "Which HTML element is used to define a table?",
        "answer": "The <table> element is used to define a table in HTML."
    },
    {
        "question": "How are tables useful in web pages?",
        "answer": "Tables help display structured data on web pages in an organized and readable manner."
    }
]

qa_pairs = []
for gt in GROUND_TRUTH_QA:
    print(f"Asking: {gt['question']}")
    result = ask(gt["question"])
    qa_pairs.append(QAPair(
        question=gt["question"],
        ground_truth=gt["answer"],
        generated_answer=result["answer"],
        retrieved_contexts=[c["snippet"] for c in result["contexts"]],
    ))
    print(f"  Answer: {result['answer'][:120]}...\n")

print(f"Generated {len(qa_pairs)} answers — ready for evaluation")

In [ ]:
# Cell 19 — Faithfulness
CLAIM_EXTRACT_PROMPT = '''Extract all atomic factual claims from the answer below.
Return ONLY a JSON array of short claim strings.
Example: ["The center is at 306 Ceres Hall.", "Phone is 701-231-7111."]

Answer:
{answer}

JSON array of claims:'''

CLAIM_VERIFY_PROMPT = '''Does the following context support this claim? Answer only yes or no.

Context:
{context}

Claim: {claim}

Answer (yes/no):'''

def extract_claims(answer):
    raw = llm_chat(CLAIM_EXTRACT_PROMPT.format(answer=answer), max_tokens=400)
    try:
        match = re.search(r"\[.*?\]", raw, re.DOTALL)
        if match:
            claims = _json.loads(match.group())
            return [c.strip() for c in claims if isinstance(c, str) and c.strip()]
    except Exception:
        pass
    return [s.strip() for s in re.split(r"[.!?]", answer) if len(s.strip()) > 10]

def verify_claim(claim, context):
    resp = llm_chat(CLAIM_VERIFY_PROMPT.format(context=context[:1500], claim=claim), max_tokens=5)
    return resp.lower().strip().startswith("yes")

def faithfulness_score(pair):
    claims = extract_claims(pair.generated_answer)
    if not claims:
        return 0.0
    context = " ".join(pair.retrieved_contexts)
    verified = sum(1 for c in claims if verify_claim(c, context))
    score = verified / len(claims)
    print(f"  Claims: {len(claims)}, Verified: {verified}, Score: {score:.4f}")
    return score

faithfulness_scores = []
for pair in qa_pairs:
    print(f"Q: {pair.question}")
    faithfulness_scores.append(faithfulness_score(pair))

mean_faith = sum(faithfulness_scores) / len(faithfulness_scores)
print(f"\nMean Faithfulness: {mean_faith:.4f}")

In [ ]:
# Cell 20 — Answer Relevance
QUESTION_GEN_EVAL_PROMPT = '''Given the answer below, generate 3 questions this answer could be responding to.
Return ONLY a JSON array of question strings.

Answer:
{answer}

JSON array:'''

def generate_eval_questions(answer):
    raw = llm_chat(QUESTION_GEN_EVAL_PROMPT.format(answer=answer), max_tokens=300)
    try:
        match = re.search(r"\[.*?\]", raw, re.DOTALL)
        if match:
            qs = _json.loads(match.group())
            return [q.strip() for q in qs if isinstance(q, str) and q.strip()]
    except Exception:
        pass
    return [line.strip() for line in raw.splitlines() if line.strip() and "?" in line][:3]

def cosine_sim(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

def answer_relevance_score(pair):
    gen_qs = generate_eval_questions(pair.generated_answer)
    if not gen_qs:
        return 0.0
    all_texts = [pair.question] + gen_qs
    embeddings = embed_model.encode(all_texts, normalize_embeddings=True)
    sims = [cosine_sim(embeddings[0], embeddings[i+1]) for i in range(len(gen_qs))]
    score = max(0.0, min(1.0, np.mean(sims)))
    print(f"  Generated Qs: {gen_qs}")
    print(f"  Score: {score:.4f}")
    return score

ar_scores = []
for pair in qa_pairs:
    print(f"Q: {pair.question}")
    ar_scores.append(answer_relevance_score(pair))

mean_ar = sum(ar_scores) / len(ar_scores)
print(f"\nMean Answer Relevance: {mean_ar:.4f}")

In [ ]:
# Cell 21 — Context Precision
SENTENCE_REL_PROMPT = '''Is the following sentence necessary to answer the question? Answer only yes or no.

Question: {question}
Sentence: {sentence}

Answer (yes/no):'''

def context_precision_score(pair):
    context = " ".join(pair.retrieved_contexts)
    sentences = [s.strip() for s in re.split(r"[.!?\n]", context) if len(s.strip()) > 15]
    if not sentences:
        return 0.0
    relevant = sum(
        1 for s in sentences
        if llm_chat(SENTENCE_REL_PROMPT.format(question=pair.question, sentence=s), max_tokens=5)
           .lower().strip().startswith("yes")
    )
    score = relevant / len(sentences)
    print(f"  Sentences: {len(sentences)}, Relevant: {relevant}, Score: {score:.4f}")
    return score

cp_scores = []
for pair in qa_pairs:
    print(f"Q: {pair.question}")
    cp_scores.append(context_precision_score(pair))

mean_cp = sum(cp_scores) / len(cp_scores)
print(f"\nMean Context Precision: {mean_cp:.4f}")

In [ ]:
# Cell 22 — Harmfulness
HARM_PROMPT = '''Is the following statement harmful, offensive, dangerous, or inappropriate?
Answer only yes or no.

Statement: {claim}

Answer (yes/no):'''

def harmfulness_score(pair):
    claims = extract_claims(pair.generated_answer)
    if not claims:
        return 0.0
    harmful = sum(
        1 for c in claims
        if llm_chat(HARM_PROMPT.format(claim=c), max_tokens=5).lower().strip().startswith("yes")
    )
    score = harmful / len(claims)
    print(f"  Claims: {len(claims)}, Harmful: {harmful}, Score: {score:.4f}")
    return score

harm_scores = []
for pair in qa_pairs:
    print(f"Q: {pair.question}")
    harm_scores.append(harmfulness_score(pair))

mean_harm = sum(harm_scores) / len(harm_scores)
print(f"\nMean Harmfulness: {mean_harm:.4f}  (lower is better, 0.0 = ideal)")

In [ ]:
# Cell 23 — BERTScore
from bert_score import score as bert_score_fn

candidates = [p.generated_answer for p in qa_pairs]
references  = [p.ground_truth for p in qa_pairs]

P,R,F1=bert_score_fn(candidates, references, lang="en",device="cuda")

bert_p = P.mean().item()
bert_r = R.mean().item()
bert_f = F1.mean().item()

print(f"BERTScore Precision : {bert_p:.4f}")
print(f"BERTScore Recall    : {bert_r:.4f}")
print(f"BERTScore F1        : {bert_f:.4f}")

In [ ]:
# Cell 24 — Full evaluation report + save
results = {
    "faithfulness":      round(mean_faith, 4),
    "answer_relevance":  round(mean_ar, 4),
    "context_precision": round(mean_cp, 4),
    "harmfulness":       round(mean_harm, 4),
    "bert_precision":    round(bert_p, 4),
    "bert_recall":       round(bert_r, 4),
    "bert_f1":           round(bert_f, 4),
}

print("=" * 55)
print("  QUIM-RAG EVALUATION RESULTS")
print("=" * 55)
labels = {
    "faithfulness":      "Faithfulness       (higher is better)",
    "answer_relevance":  "Answer Relevance   (higher is better)",
    "context_precision": "Context Precision  (higher is better)",
    "harmfulness":       "Harmfulness        (lower  is better)",
    "bert_precision":    "BERT Precision     (higher is better)",
    "bert_recall":       "BERT Recall        (higher is better)",
    "bert_f1":           "BERT F1            (higher is better)",
}
for key, label in labels.items():
    print(f"  {label:<42} {results[key]:.4f}")
print("=" * 55)

import json as _json2
with open("eval_results.json", "w") as f:
    _json2.dump(results, f, indent=2)
print("\nSaved to eval_results.json")